# Training a Stable Diffusion 3.5 Medium model with LoRA

In this section, you will take the dataset that you viewed in section 1 and use that to train a Stable Diffusion 3.5 Medium model using LoRA.

The Stable Diffusion 3.5 architecture consists of several different types of models working in conjunction with each other to generate the final output.

1. Text Encoders - the model uses 3 distinct text encoders to interpret user prompts with high accuracy
   - OpenAI CLIP-L/14: A standard text-image alignment model.
   - OpenCLIP bigG/14: A larger vision-language model for better conceptual understanding.
   - Google T5-XXL: A powerful language model (approx. 5 billion parameters) that handles complex reasoning and long prompts.
2. The Diffusion Backbone (MMDiT-X) - Stable Diffusion 3.5 medium uses a MMDiT-X architecture, which is an enhanced version of MMDiT used in the large variant.
    - It consists of a transformer with 13 layers.
    - The first 12 layers feature dual attention blocks, which allow the model to process text and image data in separate "streams" before they interact.
    - It utilizes QK-normalization to maintain training stability by normalizing the attention scores within the transformer blocks.
3. Variational Autoencoder (VAE)
    - The VAE is responsible for moving between the pixel space (the actual image) and the latent space (the mathematical representation the model works with) 
    - Encoder: Compresses an image into noise latent variables for training or image-to-image tasks.
    - Decoder: Upscales the processed latents back into a visible image at the end of the generation process.

The medium variant in this example is a 2.5B parameter model.

The image below shows the SD3.5 Architecture and a comparison of the MMDiT (figure b) and MMDiT-X (figure c) blocks. MMDiT-X uses 2 separate sets of weights: one for the text and one for the image latents.

![](images/stable-diffusion-3.5-medium_architecture.png)
source: [HuggingFace](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium)

## Prerequsities

⚠️ **Note:** You may see pip errors here about not installed or incompatible packages. They are safe to ignore.

In [ ]:
!pip install -Uq "sagemaker==3.4.1" "huggingface-hub==0.36.0" "hf_transfer==0.1.9"

## This cell will restart the kernel. 

- It will pop up a box that says "the kernel has died".
- Click "OK".

## Wait for the kernel to restart before proceeding.

In [ ]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

---

First initialize some variable that will be used in the training process. In this example you will build your own container image for training, push it to Amazon Elastic Container Repository (ECR), and retrieve it later as part of a SageMaker Training job (SMTJ). SMTJs allow you to train models on managed, ephemeral infrastructure so you can focus on your training code and less on infrastructure configuration/management.

In [ ]:
import boto3
import sagemaker
from sagemaker.core.helper import session_helper

sagemaker_session = session_helper.Session()

# Initialize session and get account details
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

# Configuration
ecr_repository_name = "stablediffusion"
ecr_image_tag = "latest"
training_container_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository_name}:{ecr_image_tag}"

role = session_helper.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

# Display configuration
print(f"""\n\nAccount: {account_id} ({region})
Container: {training_container_uri}
Role: {role}
Bucket: {bucket}
Default Prefix: {default_prefix}""")

## Build your training container

In the `training` folder, there is a script named `build_and_push_to_ecr.sh`. This script takes the `Dockerfile` from the directory and builds a training container, then creates an ECR repository (if none exists), and pushes the image to the repository so it can be used for training.

The `Dockerfile` is relatively simple in this case. It is based off `pytorch/pytorch:2.7.1-cuda12.8-cudnn9-devel` a base development level Pytorch container with Nvidia CUDA support baked in. The script will also take a set of dependencies from `requirements.txt` and bake them into the image. This is done to improve container loading time, since the dependencies won't need to be installed for every training run.

⚠️ **Note:** This section will take a few minutes as all of the dependencies get downloaded and installed. Subsequent runs will be faster depending on whether you make changes to the Dockerfile.

In [ ]:
!./training/build_and_push_to_ecr.sh $ecr_repository_name $ecr_image_tag

## Upload training data

Here, you'll take the training data that you viewed in section 1 and upload it to S3. SMTJ has an available native integration with S3 that can stage the data automatically on a training instance if you choose as part of your job configuration.

You'll also set the `model_output_path` which is where SMTJ will automatically output the compressed model artifact when training completes.

In [ ]:
local_dir = "data"
prefix = "stable-diffusion"
bucket_prefix = f"s3://{bucket}/{prefix}"
training_data_s3_path = f"{bucket_prefix}/data"

print("Start - Data sync to S3.")
!aws s3 sync {local_dir} {training_data_s3_path} --exclude "*.ipynb_checkpoints*" --exclude ".ipynb_checkpoints/*" --exclude "cache*.arrow"
print("Completed - Data sync to S3.")

model_output_path = f"{bucket_prefix}/model-output"

print(f"\n\nUsing role: {role}")
print(f"Training data path: {training_data_s3_path}")
print(f"Model output path: {model_output_path}")

## Upload base model artifacts

It is a best practice to initially download your base model artifacts to S3, then point to the S3 location for your training job instead of pointing directly at the HuggingFace model hub. Doing this will reduce your job start time and reduce the likelihood of errors in the event that the hub becomes unavailable for some reason.

⚠️**Note:** Since Stable Diffusion 3.5 medium is a gated model, you will need to use a [HuggingFace API key](https://huggingface.co/docs/hub/en/security-tokens) and accept the terms on the [model page](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium) to use it.

In [ ]:
import os

HF_TOKEN = "<ENTER YOUR HF TOKEN HERE>"

assert HF_TOKEN.startswith("hf_"), "A valid HuggingFace API Key is required"

os.environ["HF_TOKEN"] = HF_TOKEN

%store HF_TOKEN

In [ ]:
model_id = "stabilityai/stable-diffusion-3.5-medium"
model_id_filesafe = model_id.replace("/","_")

#setting use_local_model to True will download the assets to S3
use_local_model = True

⚠️ **Note:** This section will take several minutes to download the model from the HuggingFace model hub (~45GB).

In [ ]:
%%time

from huggingface_hub import snapshot_download
import os
import subprocess

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

if use_local_model:
    model_local_location = f"../models/{model_id_filesafe}"
    prefix = f"{default_prefix}/" if default_prefix else ""
    model_s3_destination = f"s3://{bucket}/{prefix}models/{model_id_filesafe}/"
    
    print(f"Downloading model {model_id}")
    os.makedirs(model_local_location, exist_ok=True)
    #snapshot_download(repo_id=model_id, local_dir=model_local_location)

    subprocess.run(['hf', 'download', '--local-dir', model_local_location, model_id])
    
    print("Beginning Model Upload...")
    subprocess.run(['aws', 's3', 'sync', model_local_location, model_s3_destination, 
                   '--exclude', '.cache/*', '--exclude', '.gitattributes'])
    
    print(f"Model uploaded to:\n{model_s3_destination}")
    os.environ["model_location"] = model_s3_destination
else:
    os.environ["model_location"] = model_id

## Configure the training job

⚠️**Note:** This training setup uses Weights and Biases (WANDB) for experiment tracking and visualizing the validation responses. This requires a [WANDB API key](https://docs.wandb.ai/models/quickstart) and an account to use it. (Free tier is enough)



In [ ]:
#Enter your WANDB API key here to enable logging, otherwise leave blank.
WANDB_API_KEY = ""

This is where the majority of the training job configuration takes place. The `ModelTrainer` class of the SageMaker Python SDK breaks up training configuration into a few different components.
- **SourceCode**: allows you to bring your own training code at job creation time and reuse a base training container. This is what you are doing in this example. The code in `source_dir` will get copied to `/opt/ml/code` in the training container. Optionally you can bake the training scripts into the container, but this approach allows for faster iterations on training code for development purposes. You can also see in this example, we use it to override the `ENTRYPOINT` of the container though the `command` parameter.
- **Compute**: this is where you configure the infrastructure requirements for your job. You can select instance type, number of instances, and additional storage space if necessary. In a real environment, the `keep_alive_period_in_seconds` parameter allows you to use SageMaker Warm Pools for training, which will keep the instance warm after a training run succeeds/fails and allow a follow-up job with the same configuration to start more quickly.
- **StoppingCondition**: Max runtime of a job. This will prevent a runaway training run from never stopping.
- **OutputDataConfig**: The place where SageMaker will output the final model artifacts from the training run.
- **InputData**: Allows you to create channels for your training data and SageMaker will automatically fetch them from S3 and put them into the appropriate folders `/opt/ml/input/data/{channel}`

This example uses a recipe based approach for training, allowing for parameters to be easily reused. The folder `training/training-scripts/recipes/` has a few different options of how to configure the job. This allows you to have a recipe with different hyperparameter configurations that you can run on different instance types for example.

There is another folder `training/training-scripts/accelerate_configs/` which houses different HuggingFace Accelerate configurations for training. In this example, you are using a Distributed Data Parallel (DDP) configuration since the model will fit on a single GPU, allowing for the data to be sharded across multiple accelerators simultaneously.

Also you will notice the `batch_size` parameter is 1, due to the limited memory on the `ml.g5.12xlarge` GPUs.

In [ ]:
from sagemaker.core.training.configs import SourceCode, Compute, InputData, OutputDataConfig, StoppingCondition

# Constants
DEFAULT_VOLUME_SIZE = 30
MAX_RUNTIME_HOURS = 24
KEEP_ALIVE_HOURS = 1

training_instance_type = "ml.g5.12xlarge"

# Define environment variables
environment={
    "WANDB_API_KEY": WANDB_API_KEY
}

#Define command line arguements for trainings script and code location
config_args = {
    "config": "recipes/default-medium-g5_12x.yaml",
    "training-script": "train_text_to_image_lora.py", 
    "accelerate-config": "accelerate_configs/ddp.yaml"
}

command_args = [f"--{k} {v}" for k, v in config_args.items()]
command = f"/bin/bash base.sh {' '.join(command_args)}"

source_code = SourceCode(
    source_dir="./training/training-scripts",
    command=command
)

# Define compute resources
compute_config = Compute(
    instance_type=training_instance_type,
    instance_count=1,
    volume_size_in_gb=DEFAULT_VOLUME_SIZE,
    keep_alive_period_in_seconds=KEEP_ALIVE_HOURS * 3600
)

# Define stopping condition
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=MAX_RUNTIME_HOURS * 3600
)

# Define output config
output_config = OutputDataConfig(
    s3_output_path=model_output_path,
)

# Define training input data
training_input = InputData(
    channel_name="train",
    data_source=training_data_s3_path,
)

# Define model input location (since we are not downloading from HF at runtime)
model_input = InputData(
    channel_name="model",
    data_source=model_s3_destination,
)


The following cell will output the training recipe.

Notable parameters include:

- **lora_layers:** which layers to train
- **rank:** what rank LoRA matricies to use
- **train_batch_size:** the batch size to use for training (reduced memory consumption)
- **gradient_accumulation_steps:** simulates larger batch sizes by holding gradients for a certain number of steps
- **max_train_steps:** the max number of steps to use for training. Determines the number of epochs.
- **num_validation_images:** the number of validation images to create (limited to 1 here for speed)
- **validation_epochs:** the number of epochs between validation runs
- **validation_inference_steps:** the number of inference steps to run on each validation image generation (higher is better but takes longer)

- **use_discrete_gpu_for_validation:** This is used specifically in this example because we cannot fit the validation model and the training weights at the same time. This flag isolates a single GPU for validation, and the rest are used for training. The training script is specially curated to support this capability.

In [ ]:
with open(f"training/training-scripts/{config_args['config']}", 'r') as file:
    # Read the entire content of the file
    file_content = file.read()
    print(file_content)

With all of the configuration in place, you can now create your `ModelTrainer` instance. The `role` parameter for the job controls the permissions that the training job will assume at runtime. It will need to be able to access the input and output destinations for the training data and model.

In [ ]:
from sagemaker.train import ModelTrainer

# Create ModelTrainer
model_trainer = ModelTrainer(
    training_image=training_container_uri,
    source_code=source_code,
    compute=compute_config,
    output_data_config=output_config,
    stopping_condition=stopping_condition,
    role=role,
    environment=environment,
    base_job_name="stablediffusion-training"
)

Call `.train()` to start the job. 

⚠️**Note:** This training run should take between 10-15 minutes. Most of the time here is starting the infrastructure and downloading the container/model. More epochs will not result in a linear increase of training time. 

You can view progress and logs inside the Studio UI under `Jobs > Training` in the left navigation in addition to the scrolling logs here. If you supplied a valid WANDB key, you should see the run show up in the WANDB UI as well.

In [ ]:
# Start training
print("\nStarting training job...")
model_trainer.train(input_data_config=[training_input, model_input], wait=True)

print(f"\nTraining complete!")
print(f"Training job: {model_trainer._latest_training_job.training_job_name}")

Once training has completed, you can get the output assets from the job. This will consist of a `model.tar.gz` archive in S3 that contains just the fine-tuned adapter in this case. You could also choose to fuse the adapter back into the base model and save the entire thing which will have better performance but sacrifice the ability to run multiple adapters for inference.

In [ ]:
import sys
from utils import get_last_job_name

job_prefix = f"stablediffusion-training"

job_name = get_last_job_name(job_prefix)

job_name

The `model.tar.gz` output is in a subfolder of the `model_output_path` defined earlier, broken down by training job id.

In [ ]:
model_data=f"{model_output_path}/{job_name}/output/model.tar.gz"

model_data

Copy the LoRA adapter locally and unpack it for use in the inference section of the workshop. There will be a `.safetensors` file inside the archive as well as a final validation image to verify the functionality of the adapter from training.

In [ ]:
local_workshop_adapter_path = "lora_adapters/adapters/workshop-adapter"

!aws s3 cp $model_data ./tuned-adapter.tar.gz
!mkdir -p $local_workshop_adapter_path
!tar -xvzf tuned-adapter.tar.gz -C $local_workshop_adapter_path
!rm tuned-adapter.tar.gz

## Training is complete! Continue on to the inference section.